# Mayo Endoskopik Skor — BULGULARDAN Kör NLP Tahmini

Bu notebook, endoskopi raporunun **yalnızca BULGULAR bölümüne** bakarak Mayo Endoskopik Skorunu 
tahmin eder. TANI bölümüne — endoskopistin yazdığı skora — bakmaz.

Bu, NHI için patoloji raporundan yapılan çıkarımın endoskopi karşılığıdır:  
serbest metin klinik tanımlamalardan yapılandırılmış skorlama.

**Performans (n=656 çift):**
- Ordinal tam uyum: %61.1 (±1 içinde: %91.8)
- Binary remisyon/aktif doğruluk: %80.3 (Sens: %74.9, Spec: %83.8)

**Gereksinim:** `pip install python-docx openpyxl`

In [8]:
import re
import openpyxl
from docx import Document
from docx.table import Table
from docx.oxml.ns import qn
from collections import Counter

## 1. Dosya Yolları

In [9]:
DOCX_FILE   = 'mayotumraporlarDOC.docx'       # Tüm endoskopi raporları
EXCEL_FILE  = 'v7.xlsx'                        # Mevcut veri seti
OUTPUT_FILE = 'v7_nlp_mayo_bulgular.xlsx'      # Çıktı (yeni sütun eklenmiş)

## 2. NLP Sistemi

Mayo kriterleri (Mayo, 1987) Türkçe endoskopi terminolojisine haritalandı:

| Mayo | Kriter | Temel Türkçe göstergeler |
|------|--------|-------------------------|
| 0 | Normal / inaktif | Hiçbir enflamasyon terimi yok |
| 1 | Hafif | eritem, ödem, vasküler patern azalmış |
| 2 | Orta | erozyon, frajil, eksüda, granüler, yer yer ülsere |
| 3 | Ağır | spontan kanama, yaygın/derin ülser, sağlam mukoza yok |

In [10]:
# ── Negasyon ────────────────────────────────────────────────────────────
NEGATION = re.compile(
    r'(yok|görülmedi|izlenmedi|saptanmadı|tespit\s+edilmedi|bulunmadı)',
    re.IGNORECASE)

def is_negated(text: str, start: int, window: int = 70) -> bool:
    """Eşleşmeden önce/sonra 70 karakter içinde negasyon var mı?"""
    return bool(NEGATION.search(text[max(0, start - window): start + window]))


# ── Terminal ileum maskeleme ─────────────────────────────────────────────
# UC için sadece kolon segmentleri Mayo skoruna girer;
# terminal ileum cümleleri UC'nin Mayo değerlendirmesini etkilememeli.
TI_SENTENCE = re.compile(
    r'[^.!?]*\b(terminal\s+ileum|t[\. ]?ileum)\b[^.!?]*[.!?]',
    re.IGNORECASE)

def mask_terminal_ileum(text: str) -> str:
    return TI_SENTENCE.sub(' ', text)


# ── Özellik kalıpları ────────────────────────────────────────────────────
PATTERNS = {
    # Mayo 3 — kesin ağır hastalık göstergeleri
    'spontaneous_bleed':  re.compile(
        r'\b(spontan\s+kanama|aktif\s+kanama|serbest\s+kanama)\b', re.I),
    'no_intact_mucosa':   re.compile(
        r'(sağlam\s+mukoza\s+kalma(?:yacak|mış)|'
        r'arada\s+normal\s+mukoza\s+(?:kalma|olmay))', re.I),
    'severe_ulcer':       re.compile(
        r'\b(yaygın|derin|büyük|geniş)\s+ülser', re.I),
    'very_friable':       re.compile(r'\boldukça\s+frajil\b', re.I),

    # Ülserasyon şiddeti ayırımı
    'focal_ulcer':        re.compile(     # hafif → Mayo 2
        r'\b(yer\s+yer|milimetrik|küçük|seyrek|tek|nadir)\s+ülser', re.I),
    'general_ulcer':      re.compile(     # fokal değilse → Mayo 3
        r'\bülser(?:asyon|e|ler|li)?\b', re.I),

    # Mayo 2 göstergeleri
    'erosion':            re.compile(r'\beroz(?:yon|yonlu|yonlar)\b', re.I),
    'friable':            re.compile(r'\bfrajil(?:ite|di|dir)?\b', re.I),
    'exudate':            re.compile(
        r'\beksüda(?:syon|syonlu|tif|t|li)?\b|eksuday?\b', re.I),
    'contact_bleeding':   re.compile(
        r'(temas\s+kanam|kanamaya\s+eğilimli)', re.I),
    'granular':           re.compile(r'\bgranüler\b', re.I),
    'vascular_absent':    re.compile(
        r'vasküler\s+(?:(?:patern[ıi]|yapı(?:lar)?)\s+)?'
        r'(?:silinm|kaybolm|bozulm|seçilem|görülem)', re.I),

    # Mayo 1 göstergeleri
    'erythema':           re.compile(
        r'\b(?:eritemli|eritem|hiperemik|hiperemi)\b', re.I),
    'edema':              re.compile(r'\bödem(?:li)?\b', re.I),
    'vascular_decreased': re.compile(
        r'vasküler\s+(?:patern[ıi]\s+)?(?:azalm|belirsizleşm)', re.I),
}


def has_feature(key: str, text: str) -> bool:
    """Özellik metinde negatif olmayan şekilde geçiyor mu?"""
    m = PATTERNS[key].search(text)
    return bool(m) and not is_negated(text, m.start())


# ── Ana tahmin fonksiyonu ────────────────────────────────────────────────
def predict_mayo_from_bulgular(bulgular_text) -> int | None:
    """
    Endoskopi raporunun BULGULAR bölümünden Mayo Endoskopik Skoru tahmin eder.
    TANI bölümüne bakmaz (kör NLP).

    Döndürür: 0, 1, 2, 3 veya None (yetersiz metin)
    """
    if not bulgular_text or len(str(bulgular_text).strip()) < 20:
        return None

    text = mask_terminal_ileum(str(bulgular_text))
    h    = has_feature  # kısaltma

    # Ülserasyon şiddeti
    focal    = h('focal_ulcer', text)
    ulc_m    = PATTERNS['general_ulcer'].search(text)
    gen_ulc  = bool(ulc_m) and not is_negated(text, ulc_m.start()) if ulc_m else False
    nonfocal = gen_ulc and not focal  # fokal olmayan ülser → Mayo 3

    # ── Mayo 3 ──────────────────────────────────────────────────────────
    if (h('spontaneous_bleed', text) or
        h('severe_ulcer', text) or
        (h('no_intact_mucosa', text) and gen_ulc) or
        (h('very_friable', text) and gen_ulc) or
        nonfocal):
        return 3

    ery  = h('erythema', text)
    edm  = h('edema', text)
    fri  = h('friable', text)
    vasc = h('vascular_absent', text)

    # ── Mayo 2 ──────────────────────────────────────────────────────────
    if (h('erosion', text) or
        h('exudate', text) or
        h('contact_bleeding', text) or
        focal or                              # yer yer/milimetrik ülsere
        (fri and (ery or edm)) or
        (vasc and fri) or
        (h('granular', text) and fri)):
        return 2

    # ── Mayo 1 ──────────────────────────────────────────────────────────
    if (ery or edm or
        h('vascular_decreased', text) or vasc or
        h('granular', text) or fri):
        return 1

    # ── Mayo 0 ──────────────────────────────────────────────────────────
    return 0


# Hızlı test
test_cases = [
    ('Tüm kolon segmentleri normaldi.', 0),
    ('Sigmoid kolon ve rektum eritemli, vasküler patern azalmış.', 1),
    ('Rektum eritemli, ödemli, frajil, erozyonlu izlendi.', 2),
    ('Sigmoid ve rektum arada sağlam mukoza kalmayacak şekilde eritemli, ülsere izlendi.', 3),
    ('Rektumda yer yer milimetrik ülsere lezyonlar görüldü.', 2),
    ('Eritemli, ödemli, vasküler yapılar silinmiş, erozyonlu izlendi.', 2),
]
print('Test sonuçları:')
all_pass = True
for text, expected in test_cases:
    result = predict_mayo_from_bulgular(text)
    ok = result == expected
    if not ok: all_pass = False
    print(f"  {'✓' if ok else '✗'}  [{result}]  {text[:60]}")
print()
print('Tüm testler geçti ✓' if all_pass else '⚠ Bazı testler başarısız!')

Test sonuçları:
  ✓  [0]  Tüm kolon segmentleri normaldi.
  ✓  [1]  Sigmoid kolon ve rektum eritemli, vasküler patern azalmış.
  ✓  [2]  Rektum eritemli, ödemli, frajil, erozyonlu izlendi.
  ✓  [3]  Sigmoid ve rektum arada sağlam mukoza kalmayacak şekilde eri
  ✓  [2]  Rektumda yer yer milimetrik ülsere lezyonlar görüldü.
  ✓  [2]  Eritemli, ödemli, vasküler yapılar silinmiş, erozyonlu izlen

Tüm testler geçti ✓


## 3. Raporları Word Dosyasından Oku

In [11]:
doc = Document(DOCX_FILE)

# Rapor numarası: "MUAYENE TARİHİ / NO : 5.01.2022 ( 65400 )" satırından
RE_REPORT_NO = re.compile(r'\(\s*(\d+)\s*\)')

parsed_reports = []
last_report_no = None

for child in doc.element.body:
    tag = child.tag.split('}')[-1]

    if tag == 'p':
        m = RE_REPORT_NO.search(child.text or '')
        if m:
            last_report_no = int(m.group(1))

    elif tag == 'tbl':
        try:
            tbl = Table(child, doc)
            row_dict = {
                r.cells[0].text.strip().upper(): r.cells[-1].text.strip()
                for r in tbl.rows
            }
            parsed_reports.append({
                'report_no': last_report_no,
                'bulgular':  row_dict.get('BULGULAR', ''),
                'tani':      row_dict.get('TANI', ''),   # referans için sakla, NLP kullanmaz
            })
        except Exception:
            pass

print(f'✓ {len(parsed_reports)} rapor okundu.')

✓ 832 rapor okundu.


## 4. Excel'i Yükle ve NLP Uygula

In [12]:
wb = openpyxl.load_workbook(EXCEL_FILE)
ws = wb.active
headers = [cell.value for cell in ws[1]]

COL_REPORT_NO   = headers.index('modified_ide_score')
COL_MAYO_MANUAL = headers.index('mayo_score')
NEW_COL         = len(headers) + 1

ws.cell(row=1, column=NEW_COL, value='nlp_mayo_from_bulgular')

# Rapor no → NLP tahmini sözlüğü
report_to_pred = {
    p['report_no']: predict_mayo_from_bulgular(p['bulgular'])
    for p in parsed_reports
    if p['report_no']
}

# Excel'e yaz
results = []
for row in ws.iter_rows(min_row=2):
    rno  = row[COL_REPORT_NO].value
    pred = report_to_pred.get(int(rno)) if rno else None
    row[NEW_COL - 1].value = pred

    manual = row[COL_MAYO_MANUAL].value
    try:    manual_int = int(float(str(manual)))
    except: manual_int = None

    results.append({'manual': manual_int, 'pred': pred})

wb.save(OUTPUT_FILE)
print(f'✓ {len(results)} satır işlendi. Kaydedildi: {OUTPUT_FILE}')

✓ 832 satır işlendi. Kaydedildi: v7_nlp_mayo_bulgular.xlsx


## 5. Validasyon

In [13]:
paired = [r for r in results if r['manual'] is not None and r['pred'] is not None]
exact  = sum(1 for r in paired if r['pred'] == r['manual'])
off1   = sum(1 for r in paired if abs(r['pred'] - r['manual']) <= 1)

# Binary: remisyon (0-1) vs aktif (2-3)
binary   = [(1 if r['manual'] <= 1 else 0, 1 if r['pred'] <= 1 else 0) for r in paired]
bin_ok   = sum(1 for m, p in binary if m == p)
tp = sum(1 for m, p in binary if m == 1 and p == 1)
tn = sum(1 for m, p in binary if m == 0 and p == 0)
fp = sum(1 for m, p in binary if m == 0 and p == 1)
fn = sum(1 for m, p in binary if m == 1 and p == 0)

print('=' * 55)
print('  NLP MAYO (BULGULARDAN) — VALİDASYON')
print('=' * 55)
print(f'  Çift sayısı                      : {len(paired)}')
print()
print(f'  ORDINAL (Mayo 0–3):')
print(f'    Tam uyum (exact agreement)      : {exact}/{len(paired)} ({exact/len(paired)*100:.1f}%)')
print(f'    ±1 içinde                       : {off1}/{len(paired)} ({off1/len(paired)*100:.1f}%)')
print()
print(f'  BİNARY (Remisyon 0–1 vs Aktif 2–3):')
print(f'    Doğruluk                        : {bin_ok}/{len(paired)} ({bin_ok/len(paired)*100:.1f}%)')
print(f'    Sensitivite (remisyon tespiti)  : {tp/(tp+fn)*100:.1f}%')
print(f'    Spesifisite (aktif tespiti)     : {tn/(tn+fp)*100:.1f}%')
print(f'    PPV                             : {tp/(tp+fp)*100:.1f}%' if (tp+fp) else '')
print(f'    NPV                             : {tn/(tn+fn)*100:.1f}%' if (tn+fn) else '')

  NLP MAYO (BULGULARDAN) — VALİDASYON
  Çift sayısı                      : 656

  ORDINAL (Mayo 0–3):
    Tam uyum (exact agreement)      : 401/656 (61.1%)
    ±1 içinde                       : 602/656 (91.8%)

  BİNARY (Remisyon 0–1 vs Aktif 2–3):
    Doğruluk                        : 527/656 (80.3%)
    Sensitivite (remisyon tespiti)  : 74.9%
    Spesifisite (aktif tespiti)     : 83.8%
    PPV                             : 74.6%
    NPV                             : 84.0%


In [7]:
# Per-class dağılım
print('Mayo bazında sonuçlar:')
for target in [0, 1, 2, 3]:
    sub = [r for r in paired if r['manual'] == target]
    cor = sum(1 for r in sub if r['pred'] == target)
    preds = Counter(r['pred'] for r in sub)
    print(f'  Mayo {target}: n={len(sub):3}  ✓={cor:3} ({cor/len(sub)*100:.0f}%)  '
          f'Dağılım: {dict(sorted(preds.items()))}')

Mayo bazında sonuçlar:
  Mayo 0: n= 81  ✓= 51 (63%)  Dağılım: {0: 51, 1: 17, 2: 8, 3: 5}
  Mayo 1: n=174  ✓=106 (61%)  Dağılım: {0: 17, 1: 106, 2: 32, 3: 19}
  Mayo 2: n=159  ✓= 74 (47%)  Dağılım: {0: 1, 1: 43, 2: 74, 3: 41}
  Mayo 3: n=242  ✓=170 (70%)  Dağılım: {0: 7, 1: 14, 2: 51, 3: 170}
